# GISLR — Next-gloss prediction fused with the recognizer, and noise rejection

**What this notebook does.** Pipeline stage 2 of sign → speech (TODO §12.6), as an
offline experiment on GISLR-Sentences:

```
MediaPipe → [ recognizer (C1) + next-gloss predictor ] → gloss → English (other notebook) → TTS
```

The stages stay **separate** (user decision, 2026-09-24). A single video → text model
is filed as future work. Three questions:

1. **Can a model predict the next gloss?** Given the glosses accepted so far (e.g.
   `minemy give`), it outputs a probability for every gloss that could come next (§1–§2).
2. **Does combining that prediction with the recognizer's confidence accept signs
   better?** The user's rule: "if the prediction and the current sign confidence match,
   accept the sign" (§4–§5).
3. **Does the recognizer reject noise that goes on for a while?** Fidgeting, hands held
   up, sign-like movements (§3–§5).

**Predictors** (`sb.rescore.prior`, `sb.rescore.neural`): uniform (no prior),
unigram / bigram / trigram / 4-gram Kneser-Ney, and a small GRU LM. Fitted on the
12.1 sentence corpus (1,757 gloss sentences), **never on the sentence being scored**:

| family | fitted on | what it measures |
|---|---|---|
| `sentence-fold` | the other 4 of 5 theme-stratified folds | unseen sentences, familiar topics (**primary**) |
| `theme-out` | the other 10 themes | unseen topic, a pessimistic case |
| `in-sample` | all sentences | memorization: an **upper bound**, never a result |

**Fusion** (`sb.recognize.continuous.fuse`) runs on C1's per-frame outputs.
Segments come from D3 or D1, with the settings the continuous-eval notebook chose.
Each segment has a recognizer vote `q` over the glosses. Rules:

| rule | accepts |
|---|---|
| `none` | `argmax q` if `q_max ≥ θ`. θ=0 and no gate reproduces D3/D1 (parity-checked vs `continuous-models.md`) |
| `rescore` | `argmax(q · p^λ)` if its normalized score `≥ θ` |
| `agree` | the fused top gloss if it is in the prior's top-k **and** `q ≥ θ_lo`. `q_max ≥ θ_hi` accepts `argmax q` alone. Otherwise reject |

Each rule also gets a **noise gate**, `max_len`: a segment longer than `max_len` frames
is rejected.

**Noise** (`sb.recognize.sequences.noise`): 1–2 blocks per stream, built from other
streams' sign frames of the same signer group. They are `fidget` (40–150 frames of
spliced sign fragments), `hold` (a sign frame held with jitter) and `reverse` (a sign
played backwards).

**Protocol:** as `gislr.3.streaming.continuous-eval.ipynb`. Settings are chosen on the
5 seeded selection signers (`sentence`, noisy), and every reported number comes from the
other 16 signers. Metrics: GER (`sb.recognize.sequences.metrics`), plus **noise
false-accept rate** (the share of noise blocks that produced a sign).

**Artifacts** (`data/cache/gislr/downstream/`):

| path | what |
|---|---|
| `lm/gru_<family>_<fold>.pt` | GRU LM weights per fold |
| `forward/<kind>-<clean|noisy>/<group>/chunk_*.npz` | C1 per-frame outputs (float16) + segments + noise blocks |
| `results/lm_eval.json`, `results/demo.json` | predictor scores, the `minemy give`-style demo |
| `results/sweep_<seg>_<part>.json` | selection sweeps |
| `results/final_parts/*.json`, `results/final.json`, `results/final_table.csv` | evaluation-signer results |
| `results/noise_summary.json`, `results/assets/*.png` | noise stats, figures |

**Resumable:** every cell skips a file that already exists. `LIMIT_PER_GROUP` runs a
smoke test into `downstream_smoke/`.

**Caveats:**
- The corpus was written by Claude to templates, so even `sentence-fold` is optimistic
  about real signing. Report the gain as an upper bound for real use.
- The `control` split (random gloss sequences, no grammar) shows what the prior costs
  when the user signs something the corpus never suggests.
- The noise is synthetic, built from sign frames. Real non-signing (talking, scratching)
  is not in it.

## Setup

In [ ]:
# ============================================================
# Imports, config, streams, signer split, corpus folds, C1
# ============================================================
import itertools
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_sentences_dir
from sb.recognize import streaming as S
from sb.recognize.continuous import cache as FC
from sb.recognize.continuous import decode as CDEC
from sb.recognize.continuous import fuse as FU
from sb.recognize.continuous.train import load_run, run_dir_for
from sb.recognize.sequences import baselines as B
from sb.recognize.sequences import metrics as M
from sb.recognize.sequences import noise as NZ
from sb.recognize.sequences.corpus import load_sentences
from sb.rescore import prior as P
from sb.rescore.neural import GRULM

CONFIG = json.loads((EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.downstream.json").read_text(encoding="utf-8"))

LIMIT_PER_GROUP = None  # e.g. 20 -> quick smoke run, written to downstream_smoke/
SKIP_GRU = False        # True -> leave the GRU LM arm out (no LM training at all)

DATASET_ROOT = gislr_sentences_dir(CONFIG["dataset_version"])
OUT = CACHE_DIR / "gislr" / ("downstream" if LIMIT_PER_GROUP is None else "downstream_smoke")
FWD, LMS, RES = OUT / "forward", OUT / "lm", OUT / "results"
ASSETS, PARTS = RES / "assets", RES / "final_parts"
for d in (FWD, LMS, RES, ASSETS, PARTS):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


# ---- streams and signer split: identical to gislr.3.streaming.continuous-eval.ipynb
SEQ_FULL = pd.read_csv(DATASET_ROOT / "sequences.csv", keep_default_na=False)
SEQ = SEQ_FULL.copy()
SEQ["row"] = np.arange(len(SEQ))
SELECT, EVAL = B.signer_split(SEQ["participant_id"], CONFIG["n_selection_signers"], CONFIG["seed"])
SEQ["group"] = np.where(SEQ["participant_id"].isin(SELECT), "select", "eval")
if LIMIT_PER_GROUP is not None:
    SEQ = SEQ.groupby(["kind", "group"], group_keys=False).head(LIMIT_PER_GROUP)

# ---- label space: sequence labels index the dataset's label map
LABEL_MAP = json.loads((DATASET_ROOT / "sign_to_prediction_index_map.json").read_text(encoding="utf-8"))
IDX2GLOSS = [g for g, _ in sorted(LABEL_MAP.items(), key=lambda kv: kv[1])]
N_CLASSES = len(IDX2GLOSS)

# ---- corpus and held-out folds
SENTS = load_sentences(CONFIG["corpus_version"])
GLOSSES = sorted({g for s in SENTS for g in s.glosses})
assert set(GLOSSES) == set(IDX2GLOSS), "corpus vocabulary differs from the label map"
ids, themes = [s.id for s in SENTS], [s.theme for s in SENTS]
FOLDS = {"sentence-fold": P.sentence_folds(ids, themes, CONFIG["lm"]["n_folds"], CONFIG["seed"]),
         "theme-out": P.theme_folds(ids, themes),
         "in-sample": {i: -1 for i in ids}}  # -1 = fitted on every sentence
assert set(SEQ_FULL.loc[SEQ_FULL.kind == "sentence", "sentence_id"]) <= set(ids)
ARMS = list(CONFIG["lm"]["ngram_orders"]) + ([] if SKIP_GRU else ["gru"])

# ---- C1 and the segmenter settings the continuous eval chose (collapsed repeats)
RUN_DIR = run_dir_for(CONFIG["run"])
MODEL, CK = load_run(RUN_DIR, DEVICE)
NULL = int(CK["null_index"])
assert NULL == N_CLASSES and [g for g, _ in sorted(CK["sign2idx"].items(), key=lambda kv: kv[1])] == IDX2GLOSS
FEATS = B.StreamFeatures.build(DATASET_ROOT, SEQ_FULL, np.asarray(CK["landmarks"]), CK["coords"])
CEVAL = json.loads((CACHE_DIR / CONFIG["continuous_eval_results"] / "final.json").read_text())
ch = pd.DataFrame(CEVAL["chosen"])
ch = ch[(ch.run == CONFIG["run"]) & (ch.collapse == CONFIG["collapse"])].set_index("decoder")
SEGMENTERS = {"D3": {"nu": float(ch.loc["D3", "nu"]), "min_len": int(ch.loc["D3", "min_len"])},
              "D1": {"beta": float(ch.loc["D1", "beta"]), "min_mass": float(ch.loc["D1", "min_mass"])}}


def segments(name: str, gp: np.ndarray, bp: np.ndarray) -> list:
    s = SEGMENTERS[name]
    if name == "D3":
        return FU.segments_d3(gp, s["nu"], s["min_len"], NULL)
    return FU.segments_d1(gp, bp, s["beta"], s["min_mass"], NULL)


print(f"dataset: {DATASET_ROOT}\nout: {OUT}\nrun {CONFIG['run']}: {RUN_DIR.name} · segmenters {SEGMENTERS}")
print(f"corpus: {len(SENTS)} sentences, {len(GLOSSES)} glosses · arms {ARMS} · device {DEVICE}")
print(SEQ.groupby(["kind", "group"]).size().unstack())

## 1. Next-gloss predictors: fitted per held-out fold, scored on the held-out sentences

Every arm is fitted once per fold of each family. The GRU LM is **trained** here (tiny:
64-d embeddings, 128-d GRU, early stopping on 10% of its training sentences, CPU). Its
weights are cached per fold. Metrics come from `sb.rescore.prior.next_gloss_metrics`
and are pooled over folds:
- **perplexity**, over every position including end-of-sentence;
- **top-k**: is the true next gloss among the k most likely? `first_` is the first
  gloss of a sentence (no history), `later_` is every other position;
- **MRR**, the mean reciprocal rank of the true next gloss.

In [ ]:
# ============================================================
# Fit every predictor per fold; pooled intrinsic scores
# ============================================================
_LM: dict = {}


def get_lm(arm: str, family: str, fold: int):
    """The `arm` predictor fitted on every corpus sentence not in `fold` of `family` (-1: all)."""
    key = (arm, family, fold)
    if key in _LM:
        return _LM[key]
    train = [s.glosses for s in SENTS if fold == -1 or FOLDS[family][s.id] != fold]
    if arm == "uniform":
        lm = P.UniformLM(GLOSSES)
    elif arm == "gru":
        g = CONFIG["lm"]["gru"]
        lm = GRULM(GLOSSES, g["emb"], g["hidden"], g["dropout"], seed=CONFIG["seed"])
        path = LMS / f"gru_{family}_{fold}.pt"
        if path.exists():
            lm.load_state_dict(torch.load(path))
        else:
            with torch.enable_grad():
                lm.fit(train, epochs=g["epochs"], lr=g["lr"], batch_size=g["batch_size"],
                       val_fraction=g["val_fraction"], patience=g["patience"])
            tmp = path.with_suffix(".tmp")
            torch.save(lm.state_dict(), tmp)
            os.replace(tmp, path)
    else:
        lm = P.NgramLM(GLOSSES, CONFIG["lm"]["ngram_orders"][arm], CONFIG["lm"]["discount"]).fit(train)
    _LM[key] = lm
    return lm


def pooled(parts: list[dict]) -> dict:
    tok = np.array([m["n_positions"] + m["n_sentences"] for m in parts], float)
    pos = np.array([m["n_positions"] for m in parts], float)
    first = np.array([m["n_sentences"] for m in parts], float)
    out = {"n_sentences": int(first.sum()),
           "perplexity": float(np.exp((np.log([m["perplexity"] for m in parts]) * tok).sum() / tok.sum())),
           "mrr": float((np.array([m["mrr"] for m in parts]) * pos).sum() / pos.sum())}
    for k in (1, 5, 10):
        for pre, w in (("", pos), ("first_", first), ("later_", pos - first)):
            v = np.array([m[f"{pre}top{k}"] for m in parts], float)
            ok = ~np.isnan(v)
            out[f"{pre}top{k}"] = float((v[ok] * w[ok]).sum() / max(w[ok].sum(), 1))
    return out


lm_path = RES / "lm_eval.json"
if not lm_path.exists():
    jobs = [(fam, arm, f) for fam in CONFIG["families"] for arm in ["uniform"] + ARMS
            for f in sorted(set(FOLDS[fam].values()))]
    per: dict = {}
    for fam, arm, f in tqdm(jobs, desc="fit + score predictors"):
        test = [s.glosses for s in SENTS if f == -1 or FOLDS[fam][s.id] == f]
        per.setdefault((fam, arm), []).append(P.next_gloss_metrics(get_lm(arm, fam, f), test))
    write_json(lm_path, [{"family": fam, "arm": arm, **pooled(parts)} for (fam, arm), parts in per.items()])
LM_EVAL = pd.DataFrame(json.loads(lm_path.read_text()))
LM_EVAL[["family", "arm", "perplexity", "top1", "top5", "top10", "first_top5", "later_top5", "mrr"]].round(3)

## 2. What does the predictor say? `minemy give` and friends

Every arm here is fitted on **all** sentences, since this is a look at the model, not a
score. For each history: the 10 most likely next tokens (`</s>` = end of sentence), and
what actually follows that history in the corpus.

In [ ]:
# ============================================================
# Top-10 next glosses for a few histories, every arm (full-corpus fits)
# ============================================================
TOP_K = 10

demo_path = RES / "demo.json"
if not demo_path.exists():
    demo = []
    for hist in CONFIG["demo_histories"]:
        n = len(hist)
        actual = Counter(s.glosses[n] if len(s.glosses) > n else P.EOS
                         for s in SENTS if list(s.glosses[:n]) == hist)
        row = {"history": " ".join(hist), "corpus_continuations": dict(actual.most_common(TOP_K))}
        for arm in ARMS:
            row[arm] = P.top_next(get_lm(arm, "in-sample", -1), hist, TOP_K)
        demo.append(row)
    write_json(demo_path, demo)
DEMO = json.loads(demo_path.read_text())
for row in DEMO:
    print(f"\n=== {row['history']} ===   corpus continues with: {row['corpus_continuations']}")
    print(pd.DataFrame({arm: [f"{g} {p:.3f}" for g, p in row[arm]] for arm in ARMS}).to_string())

## 3. C1 forward passes on clean and noisy streams (cached)

The four variants are `sentence`/`control` × `clean`/`noisy`, for both signer groups.
Noise is seeded per stream row (`noise.seed`, row), so a re-run rebuilds the same
blocks. Donor frames come from the same signer group only. C1 runs once per stream,
and its outputs go to float16 chunks.

In [ ]:
# ============================================================
# Inject noise, run C1 once per stream, cache the per-frame outputs
# ============================================================
NCFG = CONFIG["noise"]
SPEC = NZ.NoiseSpec(kinds=tuple(NCFG["kinds"]), blocks_per_stream=tuple(NCFG["blocks_per_stream"]),
                    length=tuple(NCFG["length"]), frag_len=tuple(NCFG["frag_len"]),
                    hold_jitter=NCFG["hold_jitter"], interp=NCFG["interp"])
VARIANTS = [(k, v) for k in ("sentence", "control") for v in ("clean", "noisy")]
CHUNK = CONFIG["forward_chunk"]

POOLS = {}
for grp in ("select", "eval"):
    sub = SEQ_FULL[np.isin(np.arange(len(SEQ_FULL)), SEQ.loc[SEQ.group == grp, "row"])]
    POOLS[grp] = NZ.DonorPool.from_streams([FEATS.x(i) for i in sub.index],
                                           [B.seq_arrays(r)[1] for _, r in sub.iterrows()])

todo = []
for kind, var in VARIANTS:
    for grp in ("select", "eval"):
        rows = SEQ.loc[(SEQ.kind == kind) & (SEQ.group == grp), "row"].to_numpy()
        d = FWD / f"{kind}-{var}" / grp
        d.mkdir(parents=True, exist_ok=True)
        for ci in range(0, len(rows), CHUNK):
            if not FC.chunk_path(d, ci // CHUNK).exists():
                todo.append((kind, var, grp, d, ci // CHUNK, rows[ci:ci + CHUNK]))

bar = tqdm(total=sum(len(t[-1]) for t in todo), desc="C1 forward")
for kind, var, grp, d, ci, rows in todo:
    bar.set_postfix_str(f"{kind}-{var} {grp} chunk {ci}")
    items = []
    for r in rows:
        row = SEQ_FULL.iloc[int(r)]
        x, kinds = FEATS.x(int(r)), FEATS.kind(int(r))
        labels, seg = B.seq_arrays(row)
        blocks = []
        if var == "noisy":
            rng = np.random.default_rng([NCFG["seed"], int(r)])
            x, kinds, seg, blocks = NZ.inject(x, kinds, seg, POOLS[grp], rng, SPEC)
        gp, bp = CDEC.frame_outputs(MODEL, torch.from_numpy(np.ascontiguousarray(x)))
        items.append({"row": int(r), "gp": gp, "bp": bp, "kinds": kinds, "seg": seg, "labels": labels,
                      "blocks": blocks})
        bar.update(1)
    FC.write_chunk(FC.chunk_path(d, ci), items)
bar.close()

ns_path = RES / "noise_summary.json"
if not ns_path.exists():
    summ = {}
    for grp in ("select", "eval"):
        lens = [(k, e - s) for st in FC.iter_chunks(FWD / "sentence-noisy" / grp) for s, e, k in st["blocks"]]
        summ[grp] = {k: {"n": sum(1 for kk, _ in lens if kk == k),
                         "median_len": float(np.median([n for kk, n in lens if kk == k] or [np.nan]))}
                     for k in NCFG["kinds"]}
    write_json(ns_path, summ)
print(json.loads(ns_path.read_text()))

## 4. Sweep on the selection signers (`sentence`, noisy)

Parts, each written to its own file:
- `none`: `θ` × the full `max_len` grid, per segmenter. This is the recognizer alone
  with a confidence floor and a noise gate.
- one part per (segmenter, family, arm): `rescore` and `agree` × `max_len_fusion`.

A stream's prior is the arm fitted **without that stream's sentence** (its fold in the
family). Control streams (§5) use the full-corpus fit, since they are not corpus
sentences.

In [ ]:
# ============================================================
# Selection sweeps: every rule setting, every arm, both segmenters
# ============================================================
PERM = np.array([GLOSSES.index(g) for g in IDX2GLOSS])  # label index -> predictor gloss position
SENT_ID = SEQ_FULL["sentence_id"].to_numpy()
KIND = SEQ_FULL["kind"].to_numpy()
_PRIORS: dict = {}


def prior_for(arm: str, family: str, row: int):
    fold = FOLDS[family].get(SENT_ID[row], -1) if KIND[row] == "sentence" else -1
    key = (arm, family, fold)
    if key not in _PRIORS:
        lm, memo = get_lm(arm, family, fold), {}

        def prior(hist, lm=lm, memo=memo):
            if hist not in memo:
                memo[hist] = lm.gloss_dist([IDX2GLOSS[c] for c in hist])[PERM]
            return memo[hist]
        _PRIORS[key] = prior
    return _PRIORS[key]


def load_streams(variant: str, group: str, seg_names) -> list[dict]:
    """Cached streams with their segments per segmenter; per-frame arrays dropped."""
    out = []
    for st in FC.iter_chunks(FWD / variant / group):
        st["segs"] = {n: segments(n, st["gp"], st["bp"]) for n in seg_names}
        del st["gp"], st["bp"]
        out.append(st)
    return out


def score(streams: list[dict], seg_name: str, rule: FU.Rule, arm: str | None, family: str | None) -> dict:
    sc, fa = [], Counter()
    for st in streams:
        pr = prior_for(arm, family, st["row"]) if rule.mode != "none" else None
        em = FU.decode_fused(st["segs"][seg_name], pr, rule)
        if CONFIG["collapse"]:
            em = S.collapse_repeats(em)
        sc.append(M.score_sequence(em, st["labels"], st["seg"], st["kinds"]))
        for s, e, k in st["blocks"]:
            hit = FU.noise_false_accepts(em, [(s, e)], CONFIG["noise"]["slack"])
            fa[f"{k}_blocks"] += 1
            fa[f"{k}_accepted"] += hit["blocks_accepted"]
    agg = M.aggregate(sc, B.frame_totals([st["kinds"] for st in streams]))
    nb = sum(v for k, v in fa.items() if k.endswith("_blocks"))
    agg["noise_fa"] = sum(v for k, v in fa.items() if k.endswith("_accepted")) / nb if nb else float("nan")
    for k in CONFIG["noise"]["kinds"]:
        agg[f"noise_fa_{k}"] = fa[f"{k}_accepted"] / fa[f"{k}_blocks"] if fa[f"{k}_blocks"] else float("nan")
    return agg


def rule_grid(mode: str, max_lens) -> list[FU.Rule]:
    g = CONFIG["rules"][mode]
    return [FU.Rule(mode=mode, max_len=ml, **dict(zip(g, vals)))
            for vals in itertools.product(*g.values()) for ml in max_lens]


KEEP = ("ger", "sub_rate", "del_rate", "ins_rate", "sentence_acc", "noise_fa",
        *[f"noise_fa_{k}" for k in CONFIG["noise"]["kinds"]])
kind_, var_ = CONFIG["selection_variant"].split("-")
parts = [(seg, "none", None, None) for seg in CONFIG["segmenters"]] + \
        [(seg, f"{fam}_{arm}", fam, arm) for seg in CONFIG["segmenters"]
         for fam in CONFIG["fusion_families"] for arm in ARMS]
parts = [p for p in parts if not (RES / f"sweep_{p[0]}_{p[1]}.json").exists()]
if parts:
    SEL = load_streams(CONFIG["selection_variant"], "select", CONFIG["segmenters"])
    jobs = [(p, r) for p in parts for r in (rule_grid("none", CONFIG["max_len"]) if p[2] is None else
                                          rule_grid("rescore", CONFIG["max_len_fusion"])
                                          + rule_grid("agree", CONFIG["max_len_fusion"]))]
    bar = tqdm(total=len(jobs), desc="selection sweep")
    for seg, name, fam, arm in parts:
        rows = []
        for (p, rule) in [j for j in jobs if j[0][:2] == (seg, name)]:
            a = score(SEL, seg, rule, arm, fam)
            rows.append({"segmenter": seg, "family": fam, "arm": arm or "none", **rule.as_dict(),
                         **{k: a[k] for k in KEEP}})
            bar.update(1)
            bar.set_postfix_str(f"{seg} {name}")
        write_json(RES / f"sweep_{seg}_{name}.json", rows)
    bar.close()
SWEEP = pd.concat([pd.DataFrame(json.loads(p.read_text())) for p in sorted(RES.glob("sweep_*.json"))],
                  ignore_index=True)
SWEEP.sort_values("ger").groupby(["segmenter", "family", "arm", "mode"], dropna=False).head(1)[
    ["segmenter", "family", "arm", "mode", "lam", "theta", "k", "theta_lo", "theta_hi", "max_len", "ger",
     "noise_fa"]].sort_values("ger").round(3)

## 5. Final: chosen settings on the evaluation signers, all four variants

These are the best setting per (segmenter, family, arm, rule) by selection GER, plus
two fixed references:
- **`baseline`**: `none`, θ=0, no gate. This is exactly the D3/D1 decoder, and its
  clean-sentence GER must reproduce `continuous-models.md` (parity assert, D3).
- **`gate only`**: the best `none` setting, i.e. recognizer confidence + noise gate
  with no prior.

In [ ]:
# ============================================================
# Evaluation signers: every chosen setting, 4 variants, parity vs the continuous eval
# ============================================================
RULE_KEYS = ("mode", "lam", "theta", "k", "theta_lo", "theta_hi", "max_len")


def to_rule(r) -> FU.Rule:
    kw = {k: r[k] for k in RULE_KEYS if k in r and not (isinstance(r[k], float) and np.isnan(r[k]))}
    if "max_len" in kw and kw["max_len"] is not None:
        kw["max_len"] = int(kw["max_len"])
    if "k" in kw:
        kw["k"] = int(kw["k"])
    return FU.Rule(**kw)


best = SWEEP.loc[SWEEP.groupby(["segmenter", "family", "arm", "mode"], dropna=False)["ger"].idxmin()]
CHOSEN = [{"label": (f"{r.segmenter} recognizer + gate" if r["mode"] == "none" else
                     f"{r.segmenter} {r.arm} {r['mode']} [{r.family}]"),
           "segmenter": r.segmenter, "family": r.family if isinstance(r.family, str) else None,
           "arm": None if r.arm == "none" else r.arm, "rule": to_rule(r)} for _, r in best.iterrows()]
for seg in CONFIG["segmenters"]:
    CHOSEN.append({"label": f"{seg} baseline", "segmenter": seg, "family": None, "arm": None, "rule": FU.Rule()})

for kind, var in VARIANTS:
    path = PARTS / f"{kind}-{var}.json"
    if path.exists():
        continue
    EV = load_streams(f"{kind}-{var}", "eval", CONFIG["segmenters"])
    rows = []
    for c in tqdm(CHOSEN, desc=f"final {kind}-{var}"):
        a = score(EV, c["segmenter"], c["rule"], c["arm"], c["family"])
        rows.append({"label": c["label"], "segmenter": c["segmenter"], "family": c["family"],
                     "arm": c["arm"] or "none", "variant": f"{kind}-{var}", **c["rule"].as_dict(),
                     **{k: a[k] for k in (*KEEP, "correct_rate", "emissions_per_sign", "latency_median")}})
    write_json(path, rows)

FINAL = pd.concat([pd.DataFrame(json.loads((PARTS / f"{k}-{v}.json").read_text())) for k, v in VARIANTS],
                  ignore_index=True)
ref = next(r["ger"] for r in CEVAL["results"] if r["model"] == CONFIG["run"] and r["mode"] == "D3"
           and r["collapse"] == CONFIG["collapse"] and r["split"] == "sentence" and r["frames"] == "with-null")
got = FINAL[(FINAL.label == "D3 baseline") & (FINAL.variant == "sentence-clean")]["ger"].item()
print(f"parity: D3 baseline sentence-clean GER {got:.4f} vs continuous eval {ref:.4f}")
if LIMIT_PER_GROUP is None:
    assert abs(got - ref) < 3e-3, "fused decoder with no prior/gate does not reproduce D3"
write_json(RES / "final.json", {"chosen": [{**{k: v for k, v in c.items() if k != "rule"}, **c["rule"].as_dict()}
                                           for c in CHOSEN],
                                "results": FINAL.to_dict(orient="records"), "segmenters": SEGMENTERS,
                                "select_signers": SELECT, "eval_signers": EVAL})
TABLE = FINAL.pivot_table(index="label", columns="variant", values="ger")
TABLE["noise_fa (sentence)"] = FINAL[FINAL.variant == "sentence-noisy"].set_index("label")["noise_fa"]
TABLE = TABLE.sort_values("sentence-noisy")
TABLE.to_csv(RES / "final_table.csv")
TABLE.round(3)

## 6. Figures

In [ ]:
# ============================================================
# GER by setting (D3, sentence-fold), and the noise-gate trade-off
# ============================================================
FINAL = pd.concat([pd.DataFrame(json.loads(p.read_text())) for p in sorted(PARTS.glob("*.json"))], ignore_index=True)
sub = FINAL[(FINAL.segmenter == "D3") & (FINAL.family.isin(["sentence-fold"]) | FINAL.family.isna())]
piv = sub.pivot_table(index="label", columns="variant", values="ger").sort_values("sentence-noisy")
ax = piv[["sentence-clean", "sentence-noisy", "control-clean", "control-noisy"]].plot.barh(figsize=(9, 0.5 * len(piv) + 2))
ax.invert_yaxis()
ax.set_xlabel("GER (evaluation signers, lower is better)")
ax.set_title("C1 + D3: recognizer alone vs fused with a next-gloss prior")
plt.tight_layout()
plt.savefig(ASSETS / "ger_by_setting.png", dpi=130)
plt.show()

none = SWEEP[(SWEEP["mode"] == "none") & (SWEEP.segmenter == "D3")].copy()
none["max_len"] = none["max_len"].fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(7, 4.5))
for ml, g in none.groupby("max_len"):
    ax.plot(g["noise_fa"], g["ger"], "o-", label="no gate" if ml == 0 else f"max_len {ml}")
ax.set_xlabel("noise false-accept rate (selection, sentence-noisy)")
ax.set_ylabel("GER")
ax.set_title("Recognizer alone: confidence floor θ (along each line) × noise gate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ASSETS / "noise_tradeoff.png", dpi=130)
plt.show()

## 7. Reading the results (fill in after the run)

Questions to answer in `docs/reports/downstream-next-gloss.md`:
- How much does the best prior lower GER on `sentence-clean` and `sentence-noisy`, in
  `sentence-fold`? How much survives `theme-out`?
- What does the same prior cost on `control` (unexpected sequences)?
- Which rule wins, `rescore` or the user's `agree`?
- How much noise does the gate reject, per kind (`fidget`/`hold`/`reverse`), and how
  many real signs does it lose (deletion rate)?
- Is the GRU LM worth it over the n-gram? The n-gram ships as a few KB of tables.